In [1]:
# !pip install -q curl_cffi lxml pandas

In [2]:
import time
import random
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests
# from google.colab import files

BASE_URL = "https://investor.id/terkini/indeks/{}"
MANUAL_PROXIES = []


def fetch_free_proxies():
    """Fetches public proxies as fallback."""
    print("    [Proxy Mode] Fetching public proxies...")
    url = "https://api.proxyscrape.com/v2/?request=displayproxies&protocol=http&timeout=5000&country=all&ssl=all&anonymity=all"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            proxies = [f"http://{line.strip()}" for line in resp.text.splitlines() if line.strip()]
            print(f"    [Proxy Mode] Got {len(proxies)} proxies.")
            return proxies
    except Exception as e:
        print(f"    [Proxy Mode] Failed to fetch proxy list: {e}")
    return []


def make_request(url, headers, proxy_pool=None):
    """Helper to make direct request first, falling back to proxies if blocked."""
    try:
        resp = requests.get(url, impersonate="chrome120", headers=headers, timeout=12)
        if resp.status_code == 200:
            return resp.text, proxy_pool
    except Exception:
        pass

    if proxy_pool is None:
        proxy_pool = MANUAL_PROXIES if MANUAL_PROXIES else fetch_free_proxies()

    attempts = 0
    max_attempts = 8
    while proxy_pool and attempts < max_attempts:
        attempts += 1
        current_proxy = random.choice(proxy_pool)
        proxies_dict = {"http": current_proxy, "https": current_proxy}
        try:
            resp = requests.get(url, impersonate="chrome120", proxies=proxies_dict, headers=headers, timeout=10)
            if resp.status_code == 200:
                return resp.text, proxy_pool
            else:
                proxy_pool.remove(current_proxy)
        except Exception:
            if current_proxy in proxy_pool:
                proxy_pool.remove(current_proxy)

    return None, proxy_pool


def parse_article_page(html_content, article_url):
    """Extracts news metadata and uses custom BeautifulSoup decompose logic for body content."""
    soup = BeautifulSoup(html_content, "html.parser")

    # 1. Headline: //h1[contains(@class,"h2")]
    title_tag = soup.find("h1", class_=lambda x: x and "h2" in x) or soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "N/A"

    # 2. Date: //span[contains(@class, "text-muted")]
    date_tag = soup.find("span", class_=lambda x: x and "text-muted" in x) or soup.find("time")
    published_date = date_tag.get_text(strip=True) if date_tag else "N/A"

    # 3. Canonical Link: //link[@rel="canonical"]
    canonical_tag = soup.find("link", rel="canonical")
    link = canonical_tag.get("href", article_url) if canonical_tag else article_url

    # 4. Category & Subcategory: //a[contains(@class, "text-theme")]
    category_tag = soup.find("a", class_=lambda x: x and "text-theme" in x) # Changed from find_all to find
    category = category_tag.get_text(strip=True) if category_tag else "N/A"

    # 5. Raw Paragraph Container Extraction & Decompose Cleaning Logic
    body_container = soup.find("div", class_=lambda x: x and "fsbody2" in x and "body-content" in x)

    paragraph = "N/A"
    if body_container:
        # Create a fresh isolated soup tree of the raw HTML container
        body_soup = BeautifulSoup(str(body_container), "html.parser")

        # Step 5a: Remove unwanted script, style, and heading tags
        for data in body_soup(['style', 'script', 'h2']):
            data.decompose()

        # Step 5b: Decompose div classes
        div_classes = [
            'showHide dbl-recommendation',
            'mar-10 mar-r mar-l ff-opensans',
            'boxdonasi mar-t-10',
            'mar-t-20',
            'mar-v-10',
            'bor-top mar-t-15 pad-v-10',
            'mb-3',
            'tradingview-widget-container',
            'row'
        ]
        for c in div_classes:
            for data in body_soup.find_all('div', {'class': c}):
                data.decompose()

        # Step 5c: Decompose p classes
        p_classes = ['mb-0']
        for c in p_classes:
            for data in body_soup.find_all('p', {'class': c}):
                data.decompose()

        # Step 5d: Extract clean text from remaining paragraphs
        paragraphs_list = body_soup.find_all('p')
        if paragraphs_list:
            paragraph = " ".join([p.get_text(strip=True) for p in paragraphs_list if p.get_text(strip=True)])
        else:
            paragraph = body_soup.get_text(separator=" ", strip=True)

    return {
        "Judul": title,
        "Detail": paragraph,
        "Tanggal": published_date,
        "Link": link,
        "Kategori": category

    }


def scrape_investor_id(start_page=1, max_pages=1):
    all_news = []
    proxy_pool = None

    headers = {
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "accept-language": "en-US,en;q=0.9,id;q=0.8",
        "referer": "https://investor.id/",
        "sec-ch-ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"Windows"',
        "sec-fetch-dest": "document",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "same-origin",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
    }

    for page in range(start_page, start_page + max_pages):
        index_url = BASE_URL.format(page)
        print(f"\n[Page {page}] Fetching Index Page: {index_url}")

        index_html, proxy_pool = make_request(index_url, headers, proxy_pool)
        if not index_html:
            print(f"  [!] Failed to load index page {page}.")
            continue

        index_soup = BeautifulSoup(index_html, "html.parser")

        # Locate article link cards
        col4_cards = index_soup.find_all("div", class_=lambda x: x and "col-4" in x)
        news_links = []
        for card in col4_cards:
            a_tag = card.find("a", href=True)
            if a_tag:
                news_links.append(a_tag["href"])

        if not news_links:
            news_links = [a["href"] for a in index_soup.find_all("a", href=True) if "/terkini/indeks" not in a["href"]]

        # Clean & deduplicate URLs
        clean_links = []
        for href in news_links:
            if not href or href == "#" or "javascript:" in href or "/terkini/indeks" in href:
                continue
            full_url = href if href.startswith("http") else "https://investor.id" + href
            if full_url not in clean_links:
                clean_links.append(full_url)

        print(f"  -> Found {len(clean_links)} news links on Page {page}.")

        # Scrape each article
        for idx, article_url in enumerate(clean_links, 1):
            print(f"    -> [{idx}/{len(clean_links)}] Scraping: {article_url}")
            article_html, proxy_pool = make_request(article_url, headers, proxy_pool)

            if article_html:
                article_data = parse_article_page(article_html, article_url)
                article_data["Page"] = page
                all_news.append(article_data)
            else:
                print(f"       [!] Skipped article due to request failure.")

            time.sleep(1)

    return pd.DataFrame(all_news)


# --- Execution Configuration ---
START_PAGE = 1
TOTAL_PAGES = 10

df = scrape_investor_id(start_page=START_PAGE, max_pages=TOTAL_PAGES)

# Show preview
print(f"\nExtraction complete! Collected {len(df)} articles.")
# display(df.head(10))

# # Save & Auto-download CSV in Google Colab
# output_file = f"investor_id_bs4_clean_{START_PAGE}_to_{START_PAGE + TOTAL_PAGES - 1}.csv"
# df.to_csv(output_file, index=False)
# print(f"Saved to '{output_file}'. Triggering auto-download...")
# files.download(output_file)


[Page 1] Fetching Index Page: https://investor.id/terkini/indeks/1


  -> Found 20 news links on Page 1.
    -> [1/20] Scraping: https://investor.id/market/463237/harga-cpo-loncat-tinggi-berkat-aksi-beli-murah


    -> [2/20] Scraping: https://investor.id/lifestyle/463235/minuman-energi-berisiko-ganggu-kesehatan-atlet


    -> [3/20] Scraping: https://investor.id/market/463234/bei-rombak-indeks-mdka-masuk-3-indeks-sekaligus


    -> [4/20] Scraping: https://investor.id/market/463233/berita-populer-heboh-bantahan-msci-hingga-dinamika-harga-emas-perhiasan


    -> [5/20] Scraping: https://investor.id/market/463232/harga-minyak-dunia-mendidih-selat-hormuz-memanas


... (201 outputs left out)

    -> [18/20] Scraping: https://investor.id/national/463038/ketua-mpr-buka-amplop-surat-presiden-usulkan-andi-amran-sulaiman-dan-sugiono-jadi-cawapres


    -> [19/20] Scraping: https://investor.id/national/463036/resmi-serahkan-dua-nama-calon-wapres-ke-mpr-prabowo-dipilih-pada-paripurna-11-desember


    -> [20/20] Scraping: https://investor.id/business/463035/jblperluas-penggunaan-truk-listrik-untuk-distribusi-ritel-di-jabodetabek



Extraction complete! Collected 200 articles.


In [3]:
# import time
# import random
# import pandas as pd
# from bs4 import BeautifulSoup
# from curl_cffi import requests
# from google.colab import files

# BASE_URL = "https://investor.id/terkini/indeks/{}"
# MANUAL_PROXIES = []


# def fetch_free_proxies():
#     """Fetches public proxies as fallback."""
#     print("    [Proxy Mode] Fetching public proxies...")
#     url = "https://api.proxyscrape.com/v2/?request=displayproxies&protocol=http&timeout=5000&country=all&ssl=all&anonymity=all"
#     try:
#         resp = requests.get(url, timeout=10)
#         if resp.status_code == 200:
#             proxies = [f"http://{line.strip()}" for line in resp.text.splitlines() if line.strip()]
#             print(f"    [Proxy Mode] Got {len(proxies)} proxies.")
#             return proxies
#     except Exception as e:
#         print(f"    [Proxy Mode] Failed to fetch proxy list: {e}")
#     return []


# def make_request(url, headers, proxy_pool=None):
#     """Helper to make direct request first, falling back to proxies if blocked."""
#     try:
#         resp = requests.get(url, impersonate="chrome120", headers=headers, timeout=12)
#         if resp.status_code == 200:
#             return resp.text, proxy_pool
#     except Exception:
#         pass

#     if proxy_pool is None:
#         proxy_pool = MANUAL_PROXIES if MANUAL_PROXIES else fetch_free_proxies()

#     attempts = 0
#     max_attempts = 8
#     while proxy_pool and attempts < max_attempts:
#         attempts += 1
#         current_proxy = random.choice(proxy_pool)
#         proxies_dict = {"http": current_proxy, "https": current_proxy}
#         try:
#             resp = requests.get(url, impersonate="chrome120", proxies=proxies_dict, headers=headers, timeout=10)
#             if resp.status_code == 200:
#                 return resp.text, proxy_pool
#             else:
#                 proxy_pool.remove(current_proxy)
#         except Exception:
#             if current_proxy in proxy_pool:
#                 proxy_pool.remove(current_proxy)

#     return None, proxy_pool


# def parse_article_page(html_content, article_url):
#     """Extracts news metadata and uses custom BeautifulSoup decompose logic for body content."""
#     soup = BeautifulSoup(html_content, "html.parser")

#     # 1. Headline: //h1[contains(@class,"h2")]
#     title_tag = soup.find("h1", class_=lambda x: x and "h2" in x) or soup.find("h1")
#     title = title_tag.get_text(strip=True) if title_tag else "N/A"

#     # 2. Date: //span[contains(@class, "text-muted")]
#     date_tag = soup.find("span", class_=lambda x: x and "text-muted" in x) or soup.find("time")
#     published_date = date_tag.get_text(strip=True) if date_tag else "N/A"

#     # 3. Canonical Link: //link[@rel="canonical"]
#     canonical_tag = soup.find("link", rel="canonical")
#     link = canonical_tag.get("href", article_url) if canonical_tag else article_url

#     # 4. Category & Subcategory: //a[contains(@class, "text-theme")]
#     category_tag = soup.find("a", class_=lambda x: x and "text-theme" in x) # Changed from find_all to find
#     category = category_tag.get_text(strip=True) if category_tag else "N/A"

#     # 5. Raw Paragraph Container Extraction & Decompose Cleaning Logic
#     body_container = soup.find("div", class_=lambda x: x and "fsbody2" in x and "body-content" in x)

#     paragraph = "N/A"
#     if body_container:
#         # Create a fresh isolated soup tree of the raw HTML container
#         body_soup = BeautifulSoup(str(body_container), "html.parser")

#         # Step 5a: Remove unwanted script, style, and heading tags
#         for data in body_soup(['style', 'script', 'h2']):
#             data.decompose()

#         # Step 5b: Decompose div classes
#         div_classes = [
#             'showHide dbl-recommendation',
#             'mar-10 mar-r mar-l ff-opensans',
#             'boxdonasi mar-t-10',
#             'mar-t-20',
#             'mar-v-10',
#             'bor-top mar-t-15 pad-v-10',
#             'mb-3',
#             'tradingview-widget-container',
#             'row'
#         ]
#         for c in div_classes:
#             for data in body_soup.find_all('div', {'class': c}):
#                 data.decompose()

#         # Step 5c: Decompose p classes
#         p_classes = ['mb-0']
#         for c in p_classes:
#             for data in body_soup.find_all('p', {'class': c}):
#                 data.decompose()

#         # Step 5d: Extract clean text from remaining paragraphs
#         paragraphs_list = body_soup.find_all('p')
#         if paragraphs_list:
#             paragraph = " ".join([p.get_text(strip=True) for p in paragraphs_list if p.get_text(strip=True)])
#         else:
#             paragraph = body_soup.get_text(separator=" ", strip=True)

#     return {
#         "Judul": title,
#         "Detail": paragraph,
#         "Tanggal": published_date,
#         "Link": link,
#         "Kategori": category

#     }


# def scrape_investor_id(start_page=1, max_pages=1):
#     all_news = []
#     proxy_pool = None

#     headers = {
#         "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
#         "accept-language": "en-US,en;q=0.9,id;q=0.8",
#         "referer": "https://investor.id/",
#         "sec-ch-ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
#         "sec-ch-ua-mobile": "?0",
#         "sec-ch-ua-platform": '"Windows"',
#         "sec-fetch-dest": "document",
#         "sec-fetch-mode": "navigate",
#         "sec-fetch-site": "same-origin",
#         "sec-fetch-user": "?1",
#         "upgrade-insecure-requests": "1",
#     }

#     for page in range(start_page, start_page + max_pages):
#         if page % 2 != 0: # Skip odd pages
#             print(f"\n[Page {page}] Skipping odd page.")
#             continue

#         index_url = BASE_URL.format(page)
#         print(f"\n[Page {page}] Fetching Index Page: {index_url}")

#         index_html, proxy_pool = make_request(index_url, headers, proxy_pool)
#         if not index_html:
#             print(f"  [!] Failed to load index page {page}.")
#             continue

#         index_soup = BeautifulSoup(index_html, "html.parser")

#         # Locate article link cards
#         col4_cards = index_soup.find_all("div", class_=lambda x: x and "col-4" in x)
#         news_links = []
#         for card in col4_cards:
#             a_tag = card.find("a", href=True)
#             if a_tag:
#                 news_links.append(a_tag["href"])

#         if not news_links:
#             news_links = [a["href"] for a in index_soup.find_all("a", href=True) if "/terkini/indeks" not in a["href"]]

#         # Clean & deduplicate URLs
#         clean_links = []
#         for href in news_links:
#             if not href or href == "#" or "javascript:" in href or "/terkini/indeks" in href:
#                 continue
#             full_url = href if href.startswith("http") else "https://investor.id" + href
#             if full_url not in clean_links:
#                 clean_links.append(full_url)

#         print(f"  -> Found {len(clean_links)} news links on Page {page}.")

#         # Scrape each article
#         for idx, article_url in enumerate(clean_links, 1):
#             print(f"    -> [{idx}/{len(clean_links)}] Scraping: {article_url}")
#             article_html, proxy_pool = make_request(article_url, headers, proxy_pool)

#             if article_html:
#                 article_data = parse_article_page(article_html, article_url)
#                 article_data["Page"] = page
#                 all_news.append(article_data)
#             else:
#                 print(f"       [!] Skipped article due to request failure.")

#             time.sleep(1)

#     return pd.DataFrame(all_news)


# # --- Execution Configuration ---
# START_PAGE = 1
# TOTAL_PAGES = 10

# df = scrape_investor_id(start_page=START_PAGE, max_pages=TOTAL_PAGES)

# # Show preview
# print(f"\nExtraction complete! Collected {len(df)} articles.")
# # display(df.head(10))

# # # Save & Auto-download CSV in Google Colab
# # output_file = f"investor_id_bs4_clean_{START_PAGE}_to_{START_PAGE + TOTAL_PAGES - 1}.csv"
# # df.to_csv(output_file, index=False)
# # print(f"Saved to '{output_file}'. Triggering auto-download...")
# # files.download(output_file)

In [4]:
df

,Judul,Detail,Tanggal,Link,Kategori,Page
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market,1
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,1
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market,1
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market,1
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market,1
...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,10
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National,10
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National,10
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National,10


In [5]:
df = df[df['Detail']!= "N/A"]
df

,Judul,Detail,Tanggal,Link,Kategori,Page
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market,1
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,1
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market,1
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market,1
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market,1
...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,10
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National,10
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National,10
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National,10


In [6]:
df.drop('Page', axis=1, inplace=True)
df

/tmp/ipykernel_3425/4170430715.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop('Page', axis=1, inplace=True)


,Judul,Detail,Tanggal,Link,Kategori
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market
...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National


In [7]:
df = df.drop_duplicates()
df

,Judul,Detail,Tanggal,Link,Kategori
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market
...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National


In [8]:
import datetime

def convert_to_iso(date_str):
    months_id = {
          'Jan': 'January', 'Feb': 'February', 'Mar': 'March',
          'Apr': 'April', 'Mei': 'May', 'Jun': 'June',
          'Jul': 'July', 'Aug': 'August', 'Sep': 'September',
          'Okt': 'October', 'Nov': 'November', 'Des': 'December'
      }

    try:
        parts = date_str.split(' | ')
        date_part = parts[0].strip()
        time_part = parts[1].replace('WIB', '').strip()

        for id_month, en_month in months_id.items():
            if id_month in date_part:
                date_part = date_part.replace(id_month, en_month)
                break

        dt = datetime.datetime.strptime(f"{date_part} {time_part}", "%d %B %Y %H:%M")
        return dt.isoformat()
    except Exception:
        return None

df['tanggal_iso'] = df['Tanggal'].apply(convert_to_iso)
df

,Judul,Detail,Tanggal,Link,Kategori,tanggal_iso
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market,2026-11-25T05:39:00
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,2026-11-25T05:31:00
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market,2026-11-25T05:24:00
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market,2026-11-25T05:00:00
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market,2026-11-25T04:50:00
...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,2026-11-23T17:53:00
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National,2026-11-23T17:50:00
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National,2026-11-23T17:50:00
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National,2026-11-23T17:41:00


In [9]:
df['date'] = pd.to_datetime(df['tanggal_iso'])
df

,Judul,Detail,Tanggal,Link,Kategori,tanggal_iso,date
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",25 Nov 2026 | 05:39 WIB,https://investor.id/market/463237/harga-cpo-lo...,Market,2026-11-25T05:39:00,2026-11-25 05:39:00
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",25 Nov 2026 | 05:31 WIB,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,2026-11-25T05:31:00,2026-11-25 05:31:00
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",25 Nov 2026 | 05:24 WIB,https://investor.id/market/463234/bei-rombak-i...,Market,2026-11-25T05:24:00,2026-11-25 05:24:00
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",25 Nov 2026 | 05:00 WIB,https://investor.id/market/463233/berita-popul...,Market,2026-11-25T05:00:00,2026-11-25 05:00:00
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",25 Nov 2026 | 04:50 WIB,https://investor.id/market/463232/harga-minyak...,Market,2026-11-25T04:50:00,2026-11-25 04:50:00
...,...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",23 Nov 2026 | 17:53 WIB,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,2026-11-23T17:53:00,2026-11-23 17:53:00
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463039/prabowo-pi...,National,2026-11-23T17:50:00,2026-11-23 17:50:00
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",23 Nov 2026 | 17:50 WIB,https://investor.id/national/463038/ketua-mpr-...,National,2026-11-23T17:50:00,2026-11-23 17:50:00
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",23 Nov 2026 | 17:41 WIB,https://investor.id/national/463036/resmi-sera...,National,2026-11-23T17:41:00,2026-11-23 17:41:00


In [10]:
df['Tanggal'] = df['date']
df

,Judul,Detail,Tanggal,Link,Kategori,tanggal_iso,date
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",2026-11-25 05:39:00,https://investor.id/market/463237/harga-cpo-lo...,Market,2026-11-25T05:39:00,2026-11-25 05:39:00
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",2026-11-25 05:31:00,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,2026-11-25T05:31:00,2026-11-25 05:31:00
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",2026-11-25 05:24:00,https://investor.id/market/463234/bei-rombak-i...,Market,2026-11-25T05:24:00,2026-11-25 05:24:00
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",2026-11-25 05:00:00,https://investor.id/market/463233/berita-popul...,Market,2026-11-25T05:00:00,2026-11-25 05:00:00
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",2026-11-25 04:50:00,https://investor.id/market/463232/harga-minyak...,Market,2026-11-25T04:50:00,2026-11-25 04:50:00
...,...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",2026-11-23 17:53:00,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,2026-11-23T17:53:00,2026-11-23 17:53:00
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",2026-11-23 17:50:00,https://investor.id/national/463039/prabowo-pi...,National,2026-11-23T17:50:00,2026-11-23 17:50:00
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",2026-11-23 17:50:00,https://investor.id/national/463038/ketua-mpr-...,National,2026-11-23T17:50:00,2026-11-23 17:50:00
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",2026-11-23 17:41:00,https://investor.id/national/463036/resmi-sera...,National,2026-11-23T17:41:00,2026-11-23 17:41:00


In [11]:
df = df.drop(columns=['tanggal_iso', 'date'])
df

,Judul,Detail,Tanggal,Link,Kategori
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",2026-11-25 05:39:00,https://investor.id/market/463237/harga-cpo-lo...,Market
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",2026-11-25 05:31:00,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",2026-11-25 05:24:00,https://investor.id/market/463234/bei-rombak-i...,Market
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",2026-11-25 05:00:00,https://investor.id/market/463233/berita-popul...,Market
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",2026-11-25 04:50:00,https://investor.id/market/463232/harga-minyak...,Market
...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",2026-11-23 17:53:00,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",2026-11-23 17:50:00,https://investor.id/national/463039/prabowo-pi...,National
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",2026-11-23 17:50:00,https://investor.id/national/463038/ketua-mpr-...,National
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",2026-11-23 17:41:00,https://investor.id/national/463036/resmi-sera...,National


In [12]:
from datetime import datetime

# 1. Get current system date and time
now = datetime.now()

# 2. Format it safely for a filename (Year-Month-Day_Hour-Minute-Second)
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

In [13]:
df['Source'] = "Investor"

In [14]:
df['Crawled Date'] = now.strftime("%Y-%m-%d %H:%M:%S")
df

,Judul,Detail,Tanggal,Link,Kategori,Source,Crawled Date
0,Harga CPO Loncat Tinggi Berkat Aksi Beli Murah,"JAKARTA, investor.id– Harga kontrak crude palm...",2026-11-25 05:39:00,https://investor.id/market/463237/harga-cpo-lo...,Market,Investor,2026-11-24 22:51:36
1,Minuman Energi Berisiko Ganggu Kesehatan Atlet,"JAKARTA,investor.id- Dewan Tinju Dunia (World ...",2026-11-25 05:31:00,https://investor.id/lifestyle/463235/minuman-e...,Lifestyle,Investor,2026-11-24 22:51:36
2,"BEI Rombak Indeks, MDKA Masuk 3 Indeks Sekaligus","JAKARTA, investor.id– Bursa Efek Indonesia (BE...",2026-11-25 05:24:00,https://investor.id/market/463234/bei-rombak-i...,Market,Investor,2026-11-24 22:51:36
3,BERITA POPULER: Heboh Bantahan MSCI hingga Din...,"JAKARTA, investor.id– Sebagai mediaonlineyang ...",2026-11-25 05:00:00,https://investor.id/market/463233/berita-popul...,Market,Investor,2026-11-24 22:51:36
4,"Harga Minyak Dunia Mendidih, Selat Hormuz Memanas","HOUSTON, investor.id– Harga minyak dunia melon...",2026-11-25 04:50:00,https://investor.id/market/463232/harga-minyak...,Market,Investor,2026-11-24 22:51:36
...,...,...,...,...,...,...,...
195,"Di Tengah Tekanan Global, BI Pastikan Stabilit...","JAKARTA,investor.id– Bank Indonesia (BI) menya...",2026-11-23 17:53:00,https://investor.id/macroeconomy/463040/di-ten...,Macroeconomy,Investor,2026-11-24 22:51:36
196,"Prabowo Pilih Amran dan Sugiono Jadi Cawapres,...","JAKARTA, investor.id– Presiden Prabowo Subiant...",2026-11-23 17:50:00,https://investor.id/national/463039/prabowo-pi...,National,Investor,2026-11-24 22:51:36
197,"Ketua MPR Buka Amplop Surat Presiden, Usulkan ...","JAKARTA, investor.id –Ketua Majelis Permusyawa...",2026-11-23 17:50:00,https://investor.id/national/463038/ketua-mpr-...,National,Investor,2026-11-24 22:51:36
198,"Resmi Serahkan Dua Nama Calon Wapres ke MPR, P...","JAKARTA, investor.id– Presiden RI Prabowo Subi...",2026-11-23 17:41:00,https://investor.id/national/463036/resmi-sera...,National,Investor,2026-11-24 22:51:36


In [15]:
import re

# Define a regex pattern for Excel illegal characters
# (Control characters from 0x00 to 0x1F excluding tab, newline, and carriage return, plus others)
ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010\013\014\016-\037]')

def clean_invalid_xml_chars(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub('', val)
    return val

# Apply the cleaning function to all elements in the dataframe
df_clean = df.applymap(clean_invalid_xml_chars) if hasattr(df, 'applymap') else df.map(clean_invalid_xml_chars)

# Export the cleaned dataframe to Excel
df_clean.to_excel(f"Investor_{timestamp}.xlsx", index=False)

/tmp/ipykernel_3425/1759186903.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_clean = df.applymap(clean_invalid_xml_chars) if hasattr(df, 'applymap') else df.map(clean_invalid_xml_chars)


In [16]:
# files.download(f"Investor_{timestamp}.xlsx")